In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import gdown
import zipfile
import os
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

In [ ]:
file_id = '1-7mrmHfMxh9jvJ60b0b274wFcLpdkDz5'
destination = 'downloaded_file.zip'

gdown.download(f'https://drive.google.com/uc?id={file_id}', destination, quiet=False)

with zipfile.ZipFile(destination, 'r') as zip_ref:
     zip_ref.extractall('unzip_con')

print("Download and extraction complete.")

Downloading...
From (original): https://drive.google.com/uc?id=1-7mrmHfMxh9jvJ60b0b274wFcLpdkDz5
From (redirected): https://drive.google.com/uc?id=1-7mrmHfMxh9jvJ60b0b274wFcLpdkDz5&confirm=t&uuid=110c76f8-18f0-405d-b1ce-503849ca33d6
To: /content/downloaded_file.zip
100%|██████████| 1.09G/1.09G [00:17<00:00, 61.3MB/s]


Download and extraction complete.


In [ ]:
root_path = "/content/unzip_con/food/train"
data_dict = {
    "img_path": [],
    "label": []
}

for folder_name in os.listdir(root_path):
    folder_path = os.path.join(root_path, folder_name)

    if os.path.isdir(folder_path):
        for file_name in os.listdir(folder_path):
            file_path = os.path.join(folder_path, file_name)
            if os.path.isfile(file_path):
                data_dict["img_path"].append(file_path)
                data_dict["label"].append(folder_name)

fdf = pd.DataFrame(data_dict)
fdf["label"].unique()
fdf = fdf.replace({'omelette':1, 'hot_dog':2, 'caesar_salad':3, 'hamburger':4, 'pizza':5,
       'fish':6, 'falafel':7, 'donuts':8, 'eggs':9, 'chicken_curry':10, 'sushi':11,
       'steak':12, 'baklava':13, 'spaghetti':14, 'cheesecake':15, 'ice_cream':16,
       'lasagna':17, 'chocolate_cake':18, 'french_fries':19, 'cheese_sandwich':20,
       'chicken_wings':21})

df = pd.get_dummies(fdf, columns=["label"])
df = df.replace({False : 0, True:1})

/tmp/ipython-input-2162207989.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  fdf = fdf.replace({'omelette':1, 'hot_dog':2, 'caesar_salad':3, 'hamburger':4, 'pizza':5,
/tmp/ipython-input-2162207989.py:26: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({False : 0, True:1})


In [ ]:
df

,img_path,label_1,label_2,label_3,label_4,label_5,label_6,label_7,label_8,label_9,...,label_12,label_13,label_14,label_15,label_16,label_17,label_18,label_19,label_20,label_21
0,/content/unzip_con/food/train/omelette/C0ZBZEK...,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,/content/unzip_con/food/train/omelette/JW9IH3F...,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,/content/unzip_con/food/train/omelette/AXVY7N5...,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,/content/unzip_con/food/train/omelette/EU2263P...,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,/content/unzip_con/food/train/omelette/R392ASM...,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17547,/content/unzip_con/food/train/chicken_wings/4M...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
17548,/content/unzip_con/food/train/chicken_wings/GA...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
17549,/content/unzip_con/food/train/chicken_wings/HG...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
17550,/content/unzip_con/food/train/chicken_wings/HB...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [ ]:
from sklearn.model_selection import train_test_split
df_train, df_val = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
)

In [ ]:
def df_to_tf_dataset(
    df,
    base_dir="",
    img_size=(256,256),
    batch_size=32,
    shuffle=True,
    augment=False,
    treat_neg_one_as_zero=True,
    cache=False
):
    paths = (base_dir + df.iloc[:, 0].astype(str)).values
    labels = df.iloc[:, 1:].values.astype(np.float32)

    if treat_neg_one_as_zero:
        labels[labels == -1] = 0.0

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), reshuffle_each_iteration=True)

    def _parse(path, label):
        image_bytes = tf.io.read_file(path)
        image = tf.image.decode_jpeg(image_bytes, channels=3)
        image = tf.image.convert_image_dtype(image, tf.float32)
        image = tf.image.resize(image, img_size)

        if augment:
            image = tf.image.random_flip_left_right(image)
            image = tf.image.random_brightness(image, max_delta=0.05)
            image = tf.image.random_contrast(image, lower=0.95, upper=1.05)

        return image, label

    ds = ds.map(lambda p, l: _parse(p, l), num_parallel_calls=tf.data.AUTOTUNE)

    if cache:
        ds = ds.cache()

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = df_to_tf_dataset(df_train,
                            base_dir="",
                            img_size=(256,256),
                            batch_size=32,
                            shuffle=True,
                            augment=True,
                            treat_neg_one_as_zero=False,
                            cache=False)

val_ds = df_to_tf_dataset(df_val,
                          base_dir="",
                          img_size=(256,256),
                          batch_size=32,
                          shuffle=True,
                          augment=False,
                          treat_neg_one_as_zero=False,
                          cache=False)

In [ ]:
def SepBlock(x, filters, strides=1):
    shortcut = x
    x = layers.SeparableConv2D(filters, 3, strides=strides, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.SeparableConv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if strides == 1 and shortcut.shape[-1] == x.shape[-1]:
        x = layers.Add()([shortcut, x])
    x = layers.ReLU()(x)
    return x

def build_lightweight_cnn(input_shape=(256,256,3), num_classes=21, dropout_rate=0.3):
    inputs = keras.Input(shape=input_shape)
    x = layers.Rescaling(1./255)(inputs)

    x = layers.Conv2D(32, 3, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = SepBlock(x, 32, strides=1)
    x = SepBlock(x, 64, strides=2)
    x = SepBlock(x, 64, strides=1)
    x = SepBlock(x, 128, strides=2)
    x = SepBlock(x, 128, strides=1)
    x = SepBlock(x, 256, strides=2)
    x = SepBlock(x, 256, strides=1)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name="light_cnn")
    return model

num_classes = 21
model = build_lightweight_cnn(num_classes=num_classes)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=["accuracy", keras.metrics.TopKCategoricalAccuracy(k=3, name="top3")]
)

In [ ]:
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=22,
    )

Epoch 1/22
439/439 ━━━━━━━━━━━━━━━━━━━━ 111s 179ms/step - accuracy: 0.1142 - loss: 3.0132 - top3: 0.2687 - val_accuracy: 0.0490 - val_loss: 3.2679 - val_top3: 0.1410
Epoch 2/22
439/439 ━━━━━━━━━━━━━━━━━━━━ 84s 102ms/step - accuracy: 0.2148 - loss: 2.6628 - top3: 0.4304 - val_accuracy: 0.1316 - val_loss: 4.1339 - val_top3: 0.3042
Epoch 3/22
439/439 ━━━━━━━━━━━━━━━━━━━━ 39s 89ms/step - accuracy: 0.2891 - loss: 2.4119 - top3: 0.5386 - val_accuracy: 0.2925 - val_loss: 2.5650 - val_top3: 0.5130
Epoch 4/22
439/439 ━━━━━━━━━━━━━━━━━━━━ 44s 100ms/step - accuracy: 0.3534 - loss: 2.2326 - top3: 0.6109 - val_accuracy: 0.2048 - val_loss: 3.5850 - val_top3: 0.3896
Epoch 5/22
439/439 ━━━━━━━━━━━━━━━━━━━━ 39s 89ms/step - accuracy: 0.4207 - loss: 2.0429 - top3: 0.6828 - val_accuracy: 0.3013 - val_loss: 2.4968 - val_top3: 0.5517
Epoch 6/22
439/439 ━━━━━━━━━━━━━━━━━━━━ 45s 99ms/step - accuracy: 0.4613 - loss: 1.8965 - top3: 0.7247 - val_accuracy: 0.4332 - val_loss: 2.0959 - val_top3: 0.6639
Epoch 7/22
4

In [ ]:
model.save_weights("my_model2.weights.h5")

In [ ]:
loaded = build_lightweight_cnn(num_classes=21)

loaded.load_weights("/content/my_model2.weights.h5")

In [ ]:
test_dict = {
    "file": [],
}
for file_name in os.listdir("/content/unzip_con/food/test"):
    file_path = os.path.join("/content/unzip_con/food/test", file_name)
    if os.path.isfile(file_path):
        test_dict["file"].append(file_name)
test_df = pd.DataFrame(test_dict)
test_df = test_df.sort_values(by=['file']).reset_index(drop=True)
test_df

,file
0,005YYST06V93A.jpg
1,011VG8PFN3W2W.jpg
2,014XUHGNX7Z1M.jpg
3,015PRJQGBF6NB.jpg
4,01BRGL0J9UUGM.jpg
...,...
4271,ZY7IO6KZTZ1AC.jpg
4272,ZYXVVVQRMZS4X.jpg
4273,ZZ42OE3UW3KV5.jpg
4274,ZZBO80YPWBOKJ.jpg


In [ ]:
def df_to_tf_dataset_test(
    df,
    base_dir="",
    img_size=(256,256),
    batch_size=32,
):
    paths = (base_dir + df.iloc[:, 0].astype(str)).values
    ds = tf.data.Dataset.from_tensor_slices((paths))

    def _parse(path):
        image_bytes = tf.io.read_file(path)
        image = tf.image.decode_jpeg(image_bytes, channels=3)
        image = tf.image.convert_image_dtype(image, tf.float32)
        image = tf.image.resize(image, img_size)

        return image

    ds = ds.map(lambda p: _parse(p), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

test_ds = df_to_tf_dataset_test(df=test_df,
                            base_dir="/content/unzip_con/food/test/",
                            img_size=(256,256),
                            batch_size=32)

In [ ]:
predicted_probabilities = model.predict(test_ds)
predicted_probabilities

134/134 ━━━━━━━━━━━━━━━━━━━━ 9s 69ms/step


array([[1.07343693e-03, 2.12573461e-04, 2.13066023e-02, ...,
        3.12580913e-03, 8.04294599e-04, 7.69799459e-04],
       [4.99544200e-03, 9.61651374e-03, 6.80232123e-02, ...,
        2.04718672e-03, 3.84297175e-03, 2.76995986e-03],
       [3.35454615e-03, 5.63438190e-03, 1.19362306e-03, ...,
        4.35477635e-03, 1.17781700e-03, 6.58921665e-04],
       ...,
       [1.02540059e-03, 3.05458962e-04, 1.69857449e-04, ...,
        2.34164632e-04, 4.00444027e-04, 6.89395063e-04],
       [5.34799590e-04, 1.44966207e-05, 8.70030726e-06, ...,
        1.14212708e-06, 2.62027286e-04, 4.43215822e-06],
       [1.96318440e-02, 9.57586430e-03, 1.65034586e-03, ...,
        8.12408049e-03, 2.78150532e-02, 1.02924615e-01]], dtype=float32)

In [ ]:
def batch_to_one_hot_numpy(probability_matrix):
    probs_array = np.array(probability_matrix)
    max_indices = np.argmax(probs_array, axis=1)
    one_hot_matrix = np.eye(probs_array.shape[1])[max_indices]
    return one_hot_matrix.tolist()
ans1 = batch_to_one_hot_numpy(predicted_probabilities)
def one_hot_to_labels(one_hot_matrix):
    np_matrix = np.array(one_hot_matrix)
    zero_based_indices = np.argmax(np_matrix, axis=1)
    one_based_labels = zero_based_indices + 1
    return one_based_labels.tolist()
ans = one_hot_to_labels(ans1)

In [ ]:
label_to_int = {
    'omelette': 1, 'hot_dog': 2, 'caesar_salad': 3, 'hamburger': 4, 'pizza': 5,
    'fish': 6, 'falafel': 7, 'donuts': 8, 'eggs': 9, 'chicken_curry': 10, 'sushi': 11,
    'steak': 12, 'baklava': 13, 'spaghetti': 14, 'cheesecake': 15, 'ice_cream': 16,
    'lasagna': 17, 'chocolate_cake': 18, 'french_fries': 19, 'cheese_sandwich': 20,
    'chicken_wings': 21
}

int_to_label = {v: k for k, v in label_to_int.items()}
result_list = [int_to_label.get(label, str(label)) for label in ans]
new_column = pd.Series(result_list, name='prediction')
final_df = pd.concat([test_df, new_column], axis=1)
final_df.to_csv('submission5.csv', index=False)